In [1]:
import cv2
import kagglehub
from PIL import Image

from ultralytics import YOLO, solutions

/home/senacgoon.local/202473567/.cache/pypoetry/virtualenvs/senac-ia-uc-16-computer-vision-lhRmZcUU-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
model = YOLO("yolo26n.pt")
model_speed_detector = YOLO("yolov8n.pt")

In [24]:
from pathlib import Path

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
image_path1 = project_root / "data/videos/kaggle_dataset/test/images/IMG_20759.jpg"
image_path2 = project_root / "data/videos/kaggle_dataset/test/images/IMG_20760.jpg"
image_path3 = project_root / "data/videos/kaggle_dataset/test/images/IMG_20761.jpg"

images = []
for i in range(1, 9):
    image_path = project_root / f"data/videos/kaggle_dataset/test/images/IMG_2075{i}.jpg"
    image = cv2.imread(str(image_path))
    images.append(image)

In [ ]:
results = model(images)  # list of Results objects

In [ ]:
for result in results:
    boxes = result.boxes  # Boxes object for bounding box outputs
    masks = result.masks  # Masks object for segmentation masks outputs
    keypoints = result.keypoints  # Keypoints object for pose outputs
    probs = result.probs  # Probs object for classification outputs
    obb = result.obb  # Oriented boxes object for OBB outputs
    result.show()  # display to screen
    result.save(filename="result.jpg")  # save to disk

In [ ]:
yt_results = model("https://youtu.be/LNwODJXcvt4", stream=True)

In [ ]:
yt_results.show()

In [ ]:
results = model("https://ultralytics.com/images/bus.jpg")
results = model(
    [
        "https://ultralytics.com/images/bus.jpg",
        "https://ultralytics.com/images/zidane.jpg",
    ]
)

In [ ]:
results = model("https://ultralytics.com/images/bus.jpg")  # results list

# View results
for r in results:
    print(r.boxes)

In [33]:
names = model_speed_detector.model.names

In [36]:
names

{0: 'person',
 1: 'bicycle',
 2: 'car',
 3: 'motorcycle',
 4: 'airplane',
 5: 'bus',
 6: 'train',
 7: 'truck',
 8: 'boat',
 9: 'traffic light',
 10: 'fire hydrant',
 11: 'stop sign',
 12: 'parking meter',
 13: 'bench',
 14: 'bird',
 15: 'cat',
 16: 'dog',
 17: 'horse',
 18: 'sheep',
 19: 'cow',
 20: 'elephant',
 21: 'bear',
 22: 'zebra',
 23: 'giraffe',
 24: 'backpack',
 25: 'umbrella',
 26: 'handbag',
 27: 'tie',
 28: 'suitcase',
 29: 'frisbee',
 30: 'skis',
 31: 'snowboard',
 32: 'sports ball',
 33: 'kite',
 34: 'baseball bat',
 35: 'baseball glove',
 36: 'skateboard',
 37: 'surfboard',
 38: 'tennis racket',
 39: 'bottle',
 40: 'wine glass',
 41: 'cup',
 42: 'fork',
 43: 'knife',
 44: 'spoon',
 45: 'bowl',
 46: 'banana',
 47: 'apple',
 48: 'sandwich',
 49: 'orange',
 50: 'broccoli',
 51: 'carrot',
 52: 'hot dog',
 53: 'pizza',
 54: 'donut',
 55: 'cake',
 56: 'chair',
 57: 'couch',
 58: 'potted plant',
 59: 'bed',
 60: 'dining table',
 61: 'toilet',
 62: 'tv',
 63: 'laptop',
 64: 'mou

In [37]:
cap = cv2.VideoCapture(str(project_root / "data/videos/trafic_cars_highway_edited.mp4"))
assert cap.isOpened(), "Error opening video stream or file"

w, h, fps = (int(cap.get(x)) for x in [cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS])

In [38]:
video_writer = cv2.VideoWriter(
    str(project_root / "data/videos/speed_estimation.mp4"),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

In [39]:
line_pts = [(0, 360), (1280, 360)]

In [40]:
speed_obj = solutions.SpeedEstimator()

Ultralytics Solutions: ✅ {'source': None, 'model': None, 'classes': None, 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


In [41]:
import numpy as np

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Video frame is empty or video processing has been successfully completed.")
        break

    # SpeedEstimator.process() returns a SolutionResults object, not an ndarray.
    speed_results = speed_obj.process(frame)
    annotated_frame = getattr(speed_results, "plot_im", frame)

    if annotated_frame is None or not isinstance(annotated_frame, np.ndarray):
        annotated_frame = frame

    # Keep the output frame shape consistent with VideoWriter initialization.
    if annotated_frame.shape[:2] != (h, w):
        annotated_frame = cv2.resize(annotated_frame, (w, h))

    video_writer.write(annotated_frame)

Video frame is empty or video processing has been successfully completed.


In [42]:
cap.release()
video_writer.release()
cv2.destroyAllWindows()